In [1]:
!pip install -q tiktoken

In [3]:
import pandas as pd

# Load the IMDB dataset
df = pd.read_csv(
    "IMDB Dataset.csv",
    engine="python",
    on_bad_lines="warn"
)

# Check the original dataset
print("Original dataset shape:", df.shape)

# Display the first 5 rows
df.head()

/tmp/ipykernel_5051/3848187104.py:4: ParserWarning: Skipping line 26058: unexpected end of data

  df = pd.read_csv(


Original dataset shape: (26056, 2)


,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [4]:
print("Dataset shape:", df.shape)

print("\nSentiment distribution:")
print(df["sentiment"].value_counts())

Dataset shape: (26056, 2)

Sentiment distribution:
sentiment
positive    13052
negative    13004
Name: count, dtype: int64


In [5]:
# Select 1,500 positive and 1,500 negative reviews
positive_reviews = df[df["sentiment"] == "positive"].sample(
    n=1500, random_state=42
)

negative_reviews = df[df["sentiment"] == "negative"].sample(
    n=1500, random_state=42
)

# Combine them
df = pd.concat([positive_reviews, negative_reviews])

# Shuffle the combined dataset
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

print("Final dataset shape:", df.shape)
print("\nSentiment distribution:")
print(df["sentiment"].value_counts())

Final dataset shape: (3000, 2)

Sentiment distribution:
sentiment
negative    1500
positive    1500
Name: count, dtype: int64


In [6]:
df = df.sample(frac=1, random_state=42)

In [7]:
# Convert sentiment labels into numerical values
df["label"] = df["sentiment"].map({
    "negative": 0,
    "positive": 1
})

# Display a few examples
print(df[["sentiment", "label"]].head(10))

     sentiment  label
1801  negative      0
1190  negative      0
1817  negative      0
251   positive      1
2505  positive      1
1117  negative      0
1411  positive      1
2113  negative      0
408   positive      1
2579  negative      0


In [8]:
print("\nLabel distribution:")
print(df["label"].value_counts())


Label distribution:
label
0    1500
1    1500
Name: count, dtype: int64


In [9]:
from sklearn.model_selection import train_test_split

# Separate reviews and their labels
texts = df["review"].values
labels = df["label"].values

# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    texts,
    labels,
    test_size=0.20,
    random_state=42,
    stratify=labels
)

# Display the sizes
print("Training reviews:", len(X_train))
print("Testing reviews:", len(X_test))

Training reviews: 2400
Testing reviews: 600


In [10]:
import tiktoken

# Load the cl100k_base tokenizer
encoding = tiktoken.get_encoding("cl100k_base")

print("Tokenizer loaded successfully!")

Tokenizer loaded successfully!


Before processing all 3,000 reviews, let's test it on one review

In [11]:
# Take one review
sample_review = X_train[0]

# Convert the review into token IDs
sample_tokens = encoding.encode(sample_review)

print("Original review:")
print(sample_review)

print("\nToken IDs:")
print(sample_tokens)

print("\nNumber of tokens:", len(sample_tokens))

Original review:
First of all I have to say that I'm a huge Lucio Fulci and Dario Argento fan. Although I have not seen absolutely all of their movies, I really enjoyed almost every one I did see. I really like giallos and I thought Argento was the master of this genre, after seeing films like "Tenebre" and "Phenomena". But after I saw "New York Ripper" by Fulci, I found out that he could do pretty good giallos besides his graphic zombie movies and even outdo Argento, on a certain level.<br /><br />I love Fulci's style, and yes I love gore, but this film I think, although it has a more developed plot and characters than his other films, is not his best one. What I don't like is that it can be confusing at times, especially at the end. And the fact that we go from one suspect, to another, and then another until we even suspect the retarded little girl for a moment, I think it goes too far. I know giallos are supposed to keep you guessing until the end, and the killer should be very hard

In [12]:
MAX_LEN = 200

def tokenize_and_pad(text):
    # Convert text into token IDs
    tokens = encoding.encode(text)

    # Truncate if longer than MAX_LEN
    tokens = tokens[:MAX_LEN]

    # Pad with 0 if shorter than MAX_LEN
    tokens = tokens + [0] * (MAX_LEN - len(tokens))

    return tokens

In [13]:
sample_padded = tokenize_and_pad(sample_review)

print("Original token count:", len(sample_tokens))
print("Padded/truncated length:", len(sample_padded))
print("\nFirst 20 token IDs:")
print(sample_padded[:20])

print("\nLast 10 token IDs:")
print(sample_padded[-10:])

Original token count: 367
Padded/truncated length: 200

First 20 token IDs:
[5451, 315, 682, 358, 617, 311, 2019, 430, 358, 2846, 264, 6908, 14103, 822, 48550, 5979, 323, 423, 3370, 7793]

Last 10 token IDs:
[832, 15562, 11, 311, 2500, 11, 323, 1243, 2500, 3156]


In [14]:
# Apply tokenization and padding/truncation to all training reviews
X_train_tokens = [tokenize_and_pad(text) for text in X_train]

# Apply the same process to all testing reviews
X_test_tokens = [tokenize_and_pad(text) for text in X_test]

print("Number of training sequences:", len(X_train_tokens))
print("Number of testing sequences:", len(X_test_tokens))

print("Length of one training sequence:", len(X_train_tokens[0]))
print("Length of one testing sequence:", len(X_test_tokens[0]))

Number of training sequences: 2400
Number of testing sequences: 600
Length of one training sequence: 200
Length of one testing sequence: 200


In [15]:
import torch

print("PyTorch version:", torch.__version__)

PyTorch version: 2.11.0+cpu


In [16]:
# Convert token sequences into PyTorch tensors
X_train_tensor = torch.tensor(X_train_tokens, dtype=torch.long)
X_test_tensor = torch.tensor(X_test_tokens, dtype=torch.long)

print("X_train shape:", X_train_tensor.shape)
print("X_test shape:", X_test_tensor.shape)

X_train shape: torch.Size([2400, 200])
X_test shape: torch.Size([600, 200])


In [18]:
# Convert labels into PyTorch tensors
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

print("y_train shape:", y_train_tensor.shape)
print("y_test shape:", y_test_tensor.shape)

y_train shape: torch.Size([2400])
y_test shape: torch.Size([600])


In [19]:
import torch.nn as nn

class RNNClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim):
        super().__init__()

        # Convert token IDs into dense vectors
        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim,
            padding_idx=0
        )

        # Process the sequence using an RNN
        self.rnn = nn.RNN(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            batch_first=True
        )

        # Convert the final hidden state into 2 class scores
        self.fc = nn.Linear(hidden_dim, 2)

    def forward(self, x):

        # Token IDs → embedding vectors
        x = self.embedding(x)

        # Process the sequence
        output, hidden = self.rnn(x)

        # Take the final hidden state
        x = hidden[-1]

        # Produce class scores
        x = self.fc(x)

        return x

In [20]:
vocab_size = encoding.n_vocab

embedding_dim = 64
hidden_dim = 64

model = RNNClassifier(
    vocab_size=vocab_size,
    embedding_dim=embedding_dim,
    hidden_dim=hidden_dim
)

print(model)

RNNClassifier(
  (embedding): Embedding(100277, 64, padding_idx=0)
  (rnn): RNN(64, 64, batch_first=True)
  (fc): Linear(in_features=64, out_features=2, bias=True)
)


In [21]:
# Take 4 reviews as a small test batch
sample_batch = X_train_tensor[:4]

# Pass them through the RNN
output = model(sample_batch)

print("Input shape:", sample_batch.shape)
print("Output shape:", output.shape)
print("\nModel output:")
print(output)

Input shape: torch.Size([4, 200])
Output shape: torch.Size([4, 2])

Model output:
tensor([[ 0.2474, -0.1492],
        [-0.4878,  0.1006],
        [-0.0519, -0.0238],
        [-0.3012,  0.3471]], grad_fn=<AddmmBackward0>)


In [22]:
from torch.utils.data import TensorDataset, DataLoader

# Create datasets
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

# Create DataLoaders
batch_size = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False
)

print("Number of training batches:", len(train_loader))
print("Number of testing batches:", len(test_loader))

Number of training batches: 75
Number of testing batches: 19


In [23]:
# Loss function
criterion = nn.CrossEntropyLoss()

# Optimizer
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

print("Loss function and optimizer created successfully.")

Loss function and optimizer created successfully.


In [24]:
num_epochs = 5

for epoch in range(num_epochs):

    # Training mode
    model.train()

    total_loss = 0

    for inputs, labels in train_loader:

        # Clear previous gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(inputs)

        # Calculate loss
        loss = criterion(outputs, labels)

        # Backpropagation
        loss.backward()

        # Update model weights
        optimizer.step()

        total_loss += loss.item()

    # Average loss
    average_loss = total_loss / len(train_loader)

    print(
        f"Epoch [{epoch+1}/{num_epochs}], "
        f"Loss: {average_loss:.4f}"
    )

Epoch [1/5], Loss: 0.7050
Epoch [2/5], Loss: 0.6764
Epoch [3/5], Loss: 0.6540
Epoch [4/5], Loss: 0.6183
Epoch [5/5], Loss: 0.5661


In [25]:
# Put the model in evaluation mode
model.eval()

correct = 0
total = 0

# Disable gradient calculation because we are only testing
with torch.no_grad():

    for inputs, labels in test_loader:

        # Get model predictions
        outputs = model(inputs)

        # Select the class with the highest score
        _, predicted = torch.max(outputs, 1)

        # Count total predictions
        total += labels.size(0)

        # Count correct predictions
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total

print(f"Test Accuracy: {accuracy:.2f}%")

Test Accuracy: 51.17%


In [26]:
custom_reviews = [
    "This movie was absolutely fantastic. The acting was brilliant and I loved every minute of it.",
    "The movie was boring and disappointing. The story was weak and I almost fell asleep.",
    "The movie had some good moments, but the story was confusing and the acting was only average."
]

print("Number of custom reviews:", len(custom_reviews))

Number of custom reviews: 3


In [27]:
custom_tokens = [
    tokenize_and_pad(review)
    for review in custom_reviews
]

custom_tensor = torch.tensor(
    custom_tokens,
    dtype=torch.long
)

print("Custom input shape:", custom_tensor.shape)

Custom input shape: torch.Size([3, 200])


In [28]:
model.eval()

with torch.no_grad():
    outputs = model(custom_tensor)
    predictions = torch.argmax(outputs, dim=1)

for review, prediction in zip(custom_reviews, predictions):
    sentiment = "Positive" if prediction.item() == 1 else "Negative"

    print("\nReview:")
    print(review)
    print("Predicted sentiment:", sentiment)


Review:
This movie was absolutely fantastic. The acting was brilliant and I loved every minute of it.
Predicted sentiment: Positive

Review:
The movie was boring and disappointing. The story was weak and I almost fell asleep.
Predicted sentiment: Positive

Review:
The movie had some good moments, but the story was confusing and the acting was only average.
Predicted sentiment: Positive
